# Phase 0 — Calibration on Shared Gold Benchmark

This notebook runs the current C2 system (FT E5 + BM25 + Base Qwen, no reranker)
against the shared 240-question gold benchmark to see where we stand BEFORE any
retraining.

Output: Token F1, ROUGE-L, BERTScore, Recall@K, MRR — all with bootstrap 95% CIs.

Expected runtime on Colab L4: ~30-45 min (240 questions, ~10 s/question average).

**Prereqs:**
1. Drive mounted with the existing hukuk-rag artifacts (FT E5 checkpoint-10000 at least).
2. Shared dataset present at `/content/drive/MyDrive/hukuk-rag/data/external/shared_2026/` OR uploaded to this Colab session.
3. GPU runtime selected (T4 minimum, L4 recommended).


In [ ]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 rouge-score bert-score transformers peft bitsandbytes accelerate pyarrow tqdm

import torch
print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess, os, sys
from pathlib import Path

# Clone repo if missing — uses Colab Secrets, NOT plaintext credentials
PROJECT_ROOT = Path("/content/hukuk-rag")
if not PROJECT_ROOT.exists():
    try:
        from google.colab import userdata
        gh_token = userdata.get("GITHUB_TOKEN")
        clone_url = f"https://{gh_token}@github.com/berkay-aktas/hukuk-rag.git"
    except Exception:
        clone_url = "https://github.com/berkay-aktas/hukuk-rag.git"
    subprocess.run(["git", "clone", clone_url, str(PROJECT_ROOT)], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working dir: {os.getcwd()}")

In [ ]:
DRIVE = Path("/content/drive/MyDrive/hukuk-rag")
SHARED = DRIVE / "data" / "external" / "shared_2026"

# If shared dataset isn't on Drive yet, copy from local repo
if not SHARED.exists() or not (SHARED / "corpus.jsonl").exists():
    SHARED.mkdir(parents=True, exist_ok=True)
    local_shared = PROJECT_ROOT / "data" / "external" / "shared_2026"
    if local_shared.exists():
        import shutil
        for f in local_shared.iterdir():
            shutil.copy(f, SHARED / f.name)
        print(f"Copied {len(list(SHARED.iterdir()))} files from repo to Drive")
    else:
        print("WARNING: shared dataset missing both on Drive and in repo. Upload manually to:", SHARED)

# Verify expected artifacts
for required in [
    DRIVE / "models" / "e5-checkpoints" / "checkpoint-10000",
    SHARED / "corpus.jsonl",
    SHARED / "gold_benchmark.json",
]:
    print(f"  {'OK' if required.exists() else 'MISSING'} {required}")

In [ ]:
# Build a fresh index bundle from the shared corpus using FT E5
# Takes ~10-15 min on T4/L4 (7,579 chunks × 1024-dim).
from src.pipeline.ingest import build_indexes

INDEX_DIR = DRIVE / "indexes" / "phase0_shared_ft"
FT_E5_CKPT = str(DRIVE / "models" / "e5-checkpoints" / "checkpoint-10000")

if (INDEX_DIR / "manifest.json").exists():
    print(f"Index already built at {INDEX_DIR}, skipping. Delete to rebuild.")
else:
    manifest = build_indexes(
        inputs=str(SHARED / "corpus.jsonl"),
        output_dir=str(INDEX_DIR),
        embedding_model=FT_E5_CKPT,
        faiss_nlist=128,  # small corpus, smaller nlist
        embedding_batch_size=128,
    )
    print(manifest)

In [ ]:
# Load C2-equivalent pipeline: FT E5 + BM25 + Base Qwen, no reranker
from src.generation.rag import RagPipeline

pipeline = RagPipeline.from_paths(
    index_dir=str(INDEX_DIR),
    llm_base_model="Qwen/Qwen2.5-7B-Instruct",
    qlora_adapter=None,
    use_reranker=False,
    use_bm25=True,
    use_faiss=True,
)
print("Pipeline loaded.")
print(f"  LLM: {pipeline.llm.base_model_id}  QLoRA: {'on' if pipeline.llm.is_qlora else 'off'}")
print(f"  Reranker: {'on' if pipeline.reranker else 'off'}")

In [ ]:
# Smoke test: 1 question. If output is garbage characters, abort — do NOT run full benchmark.
test_q = "Kasten adam öldürme suçunun cezası nedir?"
resp = pipeline.answer(test_q)
print(f"Q: {test_q}\n")
print(f"A: {resp.answer}\n")
print(f"Timing: {resp.timing}\n")
print("Top-3 retrieved:")
for i, r in enumerate(resp.retrieved[:3], 1):
    print(f"  [{i}] {r.score:.4f}  {r.chunk_id}  {r.text[:80]}...")

In [ ]:
# Full benchmark on shared 240-q gold
from src.pipeline.benchmark import run_benchmark

REPORT_DIR = DRIVE / "results" / "phase0_shared_gold_c2"
summary = run_benchmark(
    pipeline=pipeline,
    benchmark_path=str(SHARED / "gold_benchmark.json"),
    output_dir=str(REPORT_DIR),
    bootstrap_iterations=1000,
)

print("\n=== SHARED GOLD (240 questions) — C2 baseline ===")
print(f"Token F1: {summary['generation']['token_f1']:.4f} "
      f"[95% CI {summary['token_f1_95ci']['ci_low']:.4f}, {summary['token_f1_95ci']['ci_high']:.4f}]")
print(f"ROUGE-L:  {summary['generation'].get('rouge_l', 0):.4f}")
if summary.get('retrieval'):
    print(f"Recall@5:  {summary['retrieval']['recall@5']:.4f}")
    print(f"Recall@10: {summary['retrieval']['recall@10']:.4f}")
    print(f"MRR:       {summary['retrieval']['mrr']:.4f}")
print(f"Faithfulness: {summary['faithfulness']}")
print(f"Total time: {summary['total_seconds']}s ({summary['mean_seconds_per_q']}s/q)")

In [ ]:
# Cross-compare: run the same pipeline on OUR own 225-q gold
own_gold_path = PROJECT_ROOT / "data" / "gold" / "gold_test_set.json"
OWN_REPORT_DIR = DRIVE / "results" / "phase0_own_gold_c2"

summary_own = run_benchmark(
    pipeline=pipeline,
    benchmark_path=str(own_gold_path),
    output_dir=str(OWN_REPORT_DIR),
    bootstrap_iterations=1000,
)

print("\n=== OWN GOLD (225 questions, audited) — C2 baseline ===")
print(f"Token F1: {summary_own['generation']['token_f1']:.4f} "
      f"[95% CI {summary_own['token_f1_95ci']['ci_low']:.4f}, {summary_own['token_f1_95ci']['ci_high']:.4f}]")
print(f"ROUGE-L:  {summary_own['generation'].get('rouge_l', 0):.4f}")
print(f"Faithfulness: {summary_own['faithfulness']}")
print(f"Total time: {summary_own['total_seconds']}s")

In [ ]:
# Side-by-side calibration summary
print("=== PHASE 0 CALIBRATION SUMMARY ===\n")
print(f"{'Metric':<25} {'Own (225q)':<20} {'Shared (240q)':<20}")
print("-" * 65)
for k in ["token_f1", "rouge_l", "bleu", "exact_match"]:
    own_v = summary_own['generation'].get(k, 0)
    sh_v  = summary['generation'].get(k, 0)
    print(f"{k:<25} {own_v:<20.4f} {sh_v:<20.4f}")
print()
print(f"Faithfulness (own):    {summary_own['faithfulness']}")
print(f"Faithfulness (shared): {summary['faithfulness']}")
if summary.get('retrieval'):
    print(f"\nRetrieval on shared (chunk-level labels):")
    for k, v in summary['retrieval'].items():
        print(f"  {k}: {v:.4f}")

## Calibration interpretation

**Token F1 on shared ≈ 0.15-0.20:** same ballpark as Berat's reported numbers. Proceed with retraining (Sprint 2).

**Token F1 on shared < 0.10:** significant generalization gap. Combine shared + Yargıtay corpora before retraining.

**Token F1 on shared > 0.25:** already competitive. Focus on demo + report polish, deprioritize retraining.

**Recall@10 on shared low:** our corpus doesn't contain enough of the shared ORICON / HGK content. Solution: combine corpora and rebuild.
